# House Price Prediction

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.impute  import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from skopt import BayesSearchCV

pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.2f}'.format

In [35]:
train_data = pd.read_csv(r"train.csv")
train_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.00,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.00,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.00,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.00,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.00,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [36]:
train_data.isna().sum()

Id                  0
MSSubClass          0
MSZoning            0
LotFrontage       259
LotArea             0
Street              0
Alley            1369
LotShape            0
LandContour         0
Utilities           0
LotConfig           0
LandSlope           0
Neighborhood        0
Condition1          0
Condition2          0
BldgType            0
HouseStyle          0
OverallQual         0
OverallCond         0
YearBuilt           0
YearRemodAdd        0
RoofStyle           0
RoofMatl            0
Exterior1st         0
Exterior2nd         0
MasVnrType        872
MasVnrArea          8
ExterQual           0
ExterCond           0
Foundation          0
BsmtQual           37
BsmtCond           37
BsmtExposure       38
BsmtFinType1       37
BsmtFinSF1          0
BsmtFinType2       38
BsmtFinSF2          0
BsmtUnfSF           0
TotalBsmtSF         0
Heating             0
HeatingQC           0
CentralAir          0
Electrical          1
1stFlrSF            0
2ndFlrSF            0
LowQualFin

In [37]:
test_data = pd.read_csv("test.csv")
test_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.00,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.00,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [38]:
test_data.shape

(1459, 80)

In [39]:
test_data.isna().sum()

Id                  0
MSSubClass          0
MSZoning            4
LotFrontage       227
LotArea             0
Street              0
Alley            1352
LotShape            0
LandContour         0
Utilities           2
LotConfig           0
LandSlope           0
Neighborhood        0
Condition1          0
Condition2          0
BldgType            0
HouseStyle          0
OverallQual         0
OverallCond         0
YearBuilt           0
YearRemodAdd        0
RoofStyle           0
RoofMatl            0
Exterior1st         1
Exterior2nd         1
MasVnrType        894
MasVnrArea         15
ExterQual           0
ExterCond           0
Foundation          0
BsmtQual           44
BsmtCond           45
BsmtExposure       44
BsmtFinType1       42
BsmtFinSF1          1
BsmtFinType2       42
BsmtFinSF2          1
BsmtUnfSF           1
TotalBsmtSF         1
Heating             0
HeatingQC           0
CentralAir          0
Electrical          0
1stFlrSF            0
2ndFlrSF            0
LowQualFin

### Remove Duplicate values(check Id)

In [40]:
duplicates = train_data['Id'].duplicated()
train_data[duplicates]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


In [41]:
train_data.drop_duplicates(inplace=True)

### Remove values with no Id

In [42]:
no_id_data = train_data[train_data['Id'].isna()]
no_id_data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


In [43]:
train_data = train_data[train_data['Id'].notna()]
train_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.00,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.00,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.00,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.00,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.00,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### Target Variables 

In [44]:
x = train_data.drop(columns = ['SalePrice' , 'Id'])
y = train_data['SalePrice']

#### Transform Y using yeo-jhonson transformation

In [45]:
y_transformer = PowerTransformer(method='yeo-johnson' , standardize=True)
y_transformed = y_transformer.fit_transform(y.to_frame())

### Important Variables

In [46]:
ordinal_cols = ['LotShape','Utilities','LandSlope','ExterQual','ExterCond',
                'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional',
                'FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']



nominal_cols = ['MSZoning','Street','Alley','Neighborhood','LotConfig','BldgType','Condition1','Condition2','HouseStyle','LandContour','RoofStyle','RoofMatl','Exterior1st' ,'Exterior2nd' ,
                'MasVnrType' , 'Foundation' , 'Heating' , 'GarageType' , 'MiscFeature' , 'SaleType' , 'SaleCondition', 'Electrical' , 'CentralAir']



impute_constant_cols = ['MSSubClass','MSZoning','Street','LotShape','LandContour','Utilities','LotConfig','LandSlope',
                        'Neighborhood','Condition1','Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st',
                        'Exterior2nd','ExterQual','ExterCond','Foundation','Heating','HeatingQC','CentralAir','Electrical',
                        'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual','Fireplaces','GarageCars',
                        'PavedDrive','SaleType','SaleCondition', 'Functional']


# cols whose na values can be filled with median
numerical_cols = ['MSSubClass','LotArea','LotFrontage','OverallQual','OverallCond','YearBuilt','YearRemodAdd',
                  'MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF','2ndFlrSF','LowQualFinSF',
                  'GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd'
                  ,'Fireplaces','GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','3SsnPorch'
                  ,'ScreenPorch','PoolArea','MiscVal','MoSold','YrSold']


# cols whose na values can be filled with default text values (like 'None' or 'No Garage')
# cols whose na values can be filled with 0
# (GarageYrBlt is kept here because if a house has no garage, a 0 is a safe placeholder before scaling/binarizing)
fill_default_cols = ['Alley','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType',
                        'GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature' ,'GarageYrBlt']


default_map = {
    'Alley' : 'No alley',
    'MasVnrType' : 'None',
    'BsmtQual' : 'No Basement',
    'BsmtCond' : 'No Basement',
    'BsmtExposure' : 'No Basement',
    'BsmtFinType1' : 'No Basement',
    'BsmtFinType2' : 'No Basement',
    'FireplaceQu' : 'No Fireplace',
    'GarageType' : 'No Garage',
    'GarageFinish' : 'No Garage',
    'GarageQual' : 'No Garage',
    'GarageCond' : 'No Garage',
    'GarageYrBlt' : 0,
    'PoolQC' : 'No Pool',
    'Fence' : 'No Fence',
    'MiscFeature' : 'None'
}


ordinal_categories = {
'LotShape' : ['Missing' , 'IR3' , 'IR2' , 'IR1' , 'Reg'],
'Utilities' : ['Missing' , 'ELO' , 'NoSeWa' , 'NoSewr', 'AllPub'],
'LandSlope' : ['Missing' , 'Sev' , 'Mod' , 'Gtl'],
'ExterQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'ExterCond' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtQual' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtCond' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtExposure' : ['No Basement' , 'No' , 'Mn' , 'Av' , 'Gd'],
'BsmtFinType1' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'BsmtFinType2' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'HeatingQC' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'KitchenQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Functional' :['Missing', 'Sal' , 'Sev' , 'Maj2' , 'Maj1' , 'Mod' , 'Min2' , 'Min1' , 'Typ'],
'FireplaceQu' : ['No Fireplace' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageFinish' : ['No Garage' , 'Unf' , 'RFn' , 'Fin'],
'GarageQual' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageCond' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'PavedDrive' : ['Missing' , 'N' , 'P' , 'Y'],
'PoolQC' : ['No Pool' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Fence' : ['No Fence' , 'MnWw' , 'GdWo' , 'MnPrv' , 'GdPrv']
}


### Fill Default value columns 

In [47]:
for col in fill_default_cols:
    default_val = default_map.get(col)
    train_data[col] = train_data[col].fillna(default_val)
    test_data[col] = test_data[col].fillna(default_val)
    print(f"Default Values of {col} filled with {default_val}")
    print(train_data[col].unique())
    print(train_data[col].unique())

Default Values of Alley filled with No alley
['No alley' 'Grvl' 'Pave']
['No alley' 'Grvl' 'Pave']
Default Values of MasVnrType filled with None
['BrkFace' 'None' 'Stone' 'BrkCmn']
['BrkFace' 'None' 'Stone' 'BrkCmn']
Default Values of BsmtQual filled with No Basement
['Gd' 'TA' 'Ex' 'No Basement' 'Fa']
['Gd' 'TA' 'Ex' 'No Basement' 'Fa']
Default Values of BsmtCond filled with No Basement
['TA' 'Gd' 'No Basement' 'Fa' 'Po']
['TA' 'Gd' 'No Basement' 'Fa' 'Po']
Default Values of BsmtExposure filled with No Basement
['No' 'Gd' 'Mn' 'Av' 'No Basement']
['No' 'Gd' 'Mn' 'Av' 'No Basement']
Default Values of BsmtFinType1 filled with No Basement
['GLQ' 'ALQ' 'Unf' 'Rec' 'BLQ' 'No Basement' 'LwQ']
['GLQ' 'ALQ' 'Unf' 'Rec' 'BLQ' 'No Basement' 'LwQ']
Default Values of BsmtFinType2 filled with No Basement
['Unf' 'BLQ' 'No Basement' 'ALQ' 'Rec' 'LwQ' 'GLQ']
['Unf' 'BLQ' 'No Basement' 'ALQ' 'Rec' 'LwQ' 'GLQ']
Default Values of FireplaceQu filled with No Fireplace
['No Fireplace' 'TA' 'Gd' 'Fa' 'Ex' '

### Numerical Pipeline

In [48]:
num_pipeline = Pipeline(
    steps=[
        ('imputer' , SimpleImputer(strategy='median'))
    ]
)

### Ordinal Pipeline

In [49]:
categories = [ordinal_categories.get(col) for col in ordinal_categories.keys()]
categories

[['Missing', 'IR3', 'IR2', 'IR1', 'Reg'],
 ['Missing', 'ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
 ['Missing', 'Sev', 'Mod', 'Gtl'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Basement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Basement', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Basement', 'No', 'Mn', 'Av', 'Gd'],
 ['No Basement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
 ['No Basement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
 ['No Fireplace', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Garage', 'Unf', 'RFn', 'Fin'],
 ['No Garage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Garage', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
 ['Missing', 'N', 'P', 'Y'],
 ['No Pool', 'Fa', 'TA', 'Gd', 'Ex'],
 ['No Fence', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']]

In [50]:
ord_pipeline = Pipeline(
    steps=[
        ('imputer' , SimpleImputer(strategy='constant' , fill_value= 'Missing')),
        ('encoder' , OrdinalEncoder(categories= categories , handle_unknown= 'use_encoded_value' , unknown_value=-1))
    ]
)

### Nominal Pipeline

In [51]:
nom_pipeline = Pipeline(
    steps= [
        ('imputer' , SimpleImputer(strategy='constant' , fill_value='Missing')),
        ('encoder' , OneHotEncoder(handle_unknown='ignore' ,sparse_output = False))
    ]
)

### Preprocessing Pipeline

In [52]:
preprocessing = ColumnTransformer(
    transformers= [
        ('num' , num_pipeline , numerical_cols),
        ('ord' , ord_pipeline , ordinal_cols),
        ('nom' , nom_pipeline , nominal_cols)
    ]
)

### Main Pipeline

In [53]:
model_pipeline = Pipeline(
    steps=[
        ('preprocessing' , preprocessing),
        ('model' , GradientBoostingRegressor(random_state=42))
    ]
)

### Hyper Parameter Tuning

In [54]:
kf = KFold(n_splits=5 , random_state=42 , shuffle=True)

In [55]:
param_distributions = {
    'model__loss' : ['squared_error', 'absolute_error', 'huber'],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__n_estimators': [100, 200, 300, 500],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__min_samples_split' : [2, 10, 20, 50],
    'model__max_depth': [3, 5, 7, 9],
    'model__min_weight_fraction_leaf' : [0.0, 0.01, 0.05],
    'model__max_features': ['sqrt', 'log2', None],
    'model__validation_fraction' : [0.1,0.15,0.2],
    'model__n_iter_no_change' : [5,10,15,20,25]
}

In [56]:
bayes_Search = BayesSearchCV(
    estimator= model_pipeline,
    search_spaces= param_distributions,
    n_iter=50,
    cv = kf,
    verbose=True,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

### Training and Finding Best Model

In [57]:
bayes_Search.fit(x , y_transformed)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\ensemble\_gb.py:672: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


,estimator,Pipeline(step...m_state=42))])
,search_spaces,"{'model__learning_rate': [0.01, 0.05, ...], 'model__loss': ['squared_error', 'absolute_error', ...], 'model__max_depth': [3, 5, ...], 'model__max_features': ['sqrt', 'log2', ...], ...}"
,optimizer_kwargs,None
,n_iter,50
,scoring,'neg_mean_squared_error'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,KFold(n_split... shuffle=True)


In [58]:
print(f"Best Parametres : {bayes_Search.best_params_}")

Best Parametres : OrderedDict([('model__learning_rate', 0.1), ('model__loss', 'huber'), ('model__max_depth', 3), ('model__max_features', 'sqrt'), ('model__min_samples_split', 50), ('model__min_weight_fraction_leaf', 0.01), ('model__n_estimators', 200), ('model__n_iter_no_change', 15), ('model__subsample', 1.0), ('model__validation_fraction', 0.15)])


In [59]:
model = bayes_Search.best_estimator_

In [60]:
test_ids = test_data['Id']
test_data.drop(columns=['Id'] , inplace = True)

In [61]:
transformed_predictions = model.predict(test_data)

actual_predictions = y_transformer.inverse_transform(transformed_predictions.reshape(-1,1))
rounded_actual_predictions = [ round(x,4) for x in actual_predictions.flatten()]

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


In [62]:
results = pd.DataFrame({
    "Id" : test_ids,
    "SalePrice" : rounded_actual_predictions
})

results.to_csv("result.csv" , index=False)